# 04 — LSTM / GRU on MFCC Sequences

Audio is inherently sequential. We feed the MFCC sequence (T time steps × 40 coefficients)
directly into recurrent networks: LSTM, GRU, and vanilla RNN — comparing all three.

In [ ]:
import sys; sys.path.insert(0, '..')

import os, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from addict import Dict

from data_classes.ravdess_dataset import RAVDESSDataset
from model_classes.rnn_model import SequenceEmotionClassifier
from utils import set_seed, compute_metrics, plot_confusion_matrix, plot_training_curves, print_report

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

with open('../config/default.yaml') as f:
    cfg = Dict(yaml.safe_load(f))

set_seed(cfg.training.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Dataset

In [ ]:
test_actors      = list(cfg.data.test_actors)
train_val_actors = [a for a in range(1, 25) if a not in test_actors]

ds_args = dict(
    mode='mfcc',
    sample_rate=cfg.data.sample_rate, duration=cfg.data.duration,
    n_mfcc=cfg.data.n_mfcc, n_mels=cfg.data.n_mels,
    n_fft=cfg.data.n_fft, hop_length=cfg.data.hop_length,
)
full_ds = RAVDESSDataset(cfg.data.data_dir, actor_ids=train_val_actors, **ds_args)
test_ds = RAVDESSDataset(cfg.data.data_dir, actor_ids=test_actors,      **ds_args)

val_n   = int(0.15*len(full_ds))
train_n = len(full_ds) - val_n
train_ds, val_ds = random_split(full_ds, [train_n, val_n],
                                 generator=torch.Generator().manual_seed(cfg.training.seed))

x, label = train_ds[0]
print(f'MFCC shape: {x.shape}  (T, n_mfcc)   Label: {label}')

## 2. Train a Single Model (helper function)

In [ ]:
nw = 0
train_loader = DataLoader(train_ds, cfg.training.batch_size, shuffle=True,  num_workers=nw)
val_loader   = DataLoader(val_ds,   cfg.training.batch_size, shuffle=False, num_workers=nw)
test_loader  = DataLoader(test_ds,  cfg.training.batch_size, shuffle=False, num_workers=nw)

def train_rnn(model_type, epochs=None):
    epochs = epochs or cfg.training.epochs
    model = SequenceEmotionClassifier(
        model_type=model_type,
        input_size=cfg.data.n_mfcc,
        hidden_size=cfg.model.rnn.hidden_size,
        num_layers=cfg.model.rnn.num_layers,
        n_classes=cfg.data.n_classes,
        dropout=cfg.model.rnn.dropout,
        bidirectional=cfg.model.rnn.bidirectional,
    ).to(device)

    weights   = full_ds.class_weights().to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.training.learning_rate,
                                  weight_decay=cfg.training.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=6, factor=0.5)

    best_val, patience_cnt = 0, 0
    tr_losses, vl_losses, tr_accs, vl_accs = [], [], [], []

    for epoch in range(1, epochs+1):
        model.train()
        tl = tc = tt = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            tl += loss.item()*len(y); tc += (out.argmax(1)==y).sum().item(); tt += len(y)
        tr_losses.append(tl/tt); tr_accs.append(tc/tt)

        model.eval()
        vl = vc = vt = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                vl += loss.item()*len(y); vc += (out.argmax(1)==y).sum().item(); vt += len(y)
        vl_losses.append(vl/vt); vl_accs.append(vc/vt)
        scheduler.step(vl/vt)

        if vl_accs[-1] > best_val:
            best_val = vl_accs[-1]
            patience_cnt = 0
            os.makedirs('../saved_models', exist_ok=True)
            torch.save(model.state_dict(), f'../saved_models/best_{model_type}.pth')
        else:
            patience_cnt += 1

        if epoch % 10 == 0:
            print(f'[{model_type.upper()}] Epoch {epoch:03d}  '
                  f'train_acc={tr_accs[-1]:.4f}  val_acc={vl_accs[-1]:.4f}  best={best_val:.4f}')

        if patience_cnt >= cfg.training.early_stopping_patience:
            print(f'Early stopping at epoch {epoch}')
            break

    return model, tr_losses, vl_losses, tr_accs, vl_accs

print('Helper function defined.')

## 3. Train LSTM

In [ ]:
lstm_model, tl_lstm, vl_lstm, ta_lstm, va_lstm = train_rnn('lstm')
plot_training_curves(tl_lstm, vl_lstm, ta_lstm, va_lstm,
                     save_path='../results/curves_lstm.png')
plt.suptitle('LSTM Training Curves'); plt.show()

## 4. Train GRU

In [ ]:
gru_model, tl_gru, vl_gru, ta_gru, va_gru = train_rnn('gru')
plot_training_curves(tl_gru, vl_gru, ta_gru, va_gru,
                     save_path='../results/curves_gru.png')
plt.suptitle('GRU Training Curves'); plt.show()

## 5. Evaluate on Test Set

In [ ]:
os.makedirs('../results', exist_ok=True)

for mt, m in [('lstm', lstm_model), ('gru', gru_model)]:
    m.load_state_dict(torch.load(f'../saved_models/best_{mt}.pth', map_location=device))
    m.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            out = m(x.to(device))
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    preds, labels = np.array(preds), np.array(labels)
    metrics = compute_metrics(labels, preds)
    print(f'\n=== {mt.upper()} ===')
    print(f'Accuracy: {metrics["accuracy"]:.4f}  F1 macro: {metrics["f1_macro"]:.4f}')
    print_report(labels, preds)
    json.dump(metrics, open(f'../results/metrics_{mt}.json','w'), indent=2)
    plot_confusion_matrix(labels, preds, title=f'{mt.upper()} — Confusion Matrix',
                          save_path=f'../results/cm_{mt}.png')
    plt.show()